In [1]:
!apt-get update -qq
!apt-get install python3.10 python3.10-distutils -qq
!wget -q https://bootstrap.pypa.io/get-pip.py
!python3.10 get-pip.py -q
!python3.10 -m pip install -q mediapipe-model-maker==0.2.1.4

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Failed to fetch https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu/dists/jammy/InRelease  Could not wait for server fd - select (11: Resource temporarily unavailable) [IP: 185.125.190.80 443]
W: Failed to fetch https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu/dists/jammy/InRelease  Could not connect to ppa.launchpadcontent.net:443 (185.125.190.80), connection timed out [IP: 185.125.190.80 443]
W: Failed to fetch https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu/dists/jammy/InRelease  Unable to connect to ppa.launchpadcontent.net:443: [IP: 185.125.190.80 443]
W: Some index files failed to download. They have been ignored, or old ones used instead.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  

In [2]:
!unzip -q -o SmartHomeCore.zip -d /content/dataset

In [5]:
%%writefile train.py
import os
import sys
import glob
import shutil
from PIL import Image
from mediapipe_model_maker import object_detector

print(f"Using Python version: {sys.version}")

# --- THE DRIVE PATH (GUI handles the connection now) ---
drive_save_path = '/content/drive/MyDrive/SmartHomeAI_Model'
os.makedirs(drive_save_path, exist_ok=True)
print(f"✅ Model will save directly to: {drive_save_path}")

# --- 1. CONSOLIDATE DATASET ---
for folder in ['valid', 'test']:
    src_folder = f"/content/dataset/{folder}"
    if os.path.exists(src_folder):
        for f in glob.glob(f"{src_folder}/*"):
            try:
                shutil.move(f, "/content/dataset/train/")
            except Exception:
                pass

# --- 2. FORMAT AND CLEAN (The Terminator) ---
def format_and_clean_dataset(directory):
    if not os.path.exists(directory):
        return

    annotations_dir = os.path.join(directory, 'Annotations')
    images_dir = os.path.join(directory, 'images')

    os.makedirs(annotations_dir, exist_ok=True)
    os.makedirs(images_dir, exist_ok=True)

    for xml_file in glob.glob(os.path.join(directory, '*.xml')):
        shutil.move(xml_file, annotations_dir)

    extensions = ['*.jpg', '*.jpeg', '*.png', '*.JPG', '*.JPEG', '*.PNG']
    for ext in extensions:
        for img_file in glob.glob(os.path.join(directory, ext)):
            shutil.move(img_file, images_dir)

    print(f"Scrubbing {directory} for corrupted files...")
    for img_path in glob.glob(os.path.join(images_dir, '*')):
        try:
            img = Image.open(img_path)
            fmt = img.format
            img.close()
            if fmt not in ['JPEG', 'PNG', 'MPO']:
                raise ValueError(f"Unsupported format: {fmt}")
        except Exception:
            os.remove(img_path)
            base_name = os.path.splitext(os.path.basename(img_path))[0]
            xml_path = os.path.join(annotations_dir, base_name + '.xml')
            if os.path.exists(xml_path):
                os.remove(xml_path)

format_and_clean_dataset('/content/dataset/train')

# --- 3. DYNAMIC SPLIT AND TRAIN ---
print("Loading Smart Home Dataset...")
full_data = object_detector.Dataset.from_pascal_voc_folder('/content/dataset/train')
train_data, validation_data = full_data.split(0.8)

print("Configuring Architecture...")
hparams = object_detector.HParams(
    epochs=100,
    batch_size=8,
    learning_rate=0.01,
    export_dir=drive_save_path  # SAVING DIRECTLY TO YOUR MOUNTED DRIVE
)

options = object_detector.ObjectDetectorOptions(
    supported_model=object_detector.SupportedModels.MOBILENET_V2,
    hparams=hparams
)

print("🚀 Starting Model Training...")
model = object_detector.ObjectDetector.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)

print("Exporting to Google Drive...")
model.export_model()
print("✅ Training Complete! Check your Google Drive.")

Overwriting train.py


In [6]:
!env MPLBACKEND=agg python3.10 train.py

2026-05-02 17:35:24.231129: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-05-02 17:35:24.231305: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-05-02 17:35:24.232714: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-02 17:35:24.240941: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
/usr/local/lib/python3.10/dist-packages/googl